In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!gdown --id 19slGAb6cPmIaDeBHvzscRO0aJTIEjb0w -O /kaggle/working/trainpval_features_all.parquet

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=19slGAb6cPmIaDeBHvzscRO0aJTIEjb0w
From (redirected): https://drive.google.com/uc?id=19slGAb6cPmIaDeBHvzscRO0aJTIEjb0w&confirm=t&uuid=403cef15-1b57-4ef7-9586-5eb606381269
To: /kaggle/working/trainpval_features_all.parquet
100%|████████████████████████████████████████| 301M/301M [00:07<00:00, 42.1MB/s]


In [3]:
!gdown --id 1nbFR8YuObo0EEdKwRW8CC1G8NZpDTCrC -O /kaggle/working/test_features_all.parquet

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1nbFR8YuObo0EEdKwRW8CC1G8NZpDTCrC
To: /kaggle/working/test_features_all.parquet
100%|█████████████████████████████████████████| 756k/756k [00:00<00:00, 101MB/s]


In [4]:
# Extract first 500,000 rows

trainvaldf = pd.read_parquet("/kaggle/working/trainpval_features_all.parquet")

testdf = pd.read_parquet("/kaggle/working/test_features_all.parquet")
df = trainvaldf.iloc[:500000]
valdf = trainvaldf.iloc[500000:]
# Optional: check shape
print(df.shape)
print(valdf.shape)
print(testdf.shape)
out_path = f"train_features_all.parquet"
df.to_parquet(
    out_path,
    index=False
)

(500000, 38)
(100000, 38)
(1000, 38)


In [5]:
FINAL_FEATURES = [  
    "loc_total","loc_comments","loc_blank","comment_ratio","avg_line_length",
    "max_line_length","indentation_std_dev",
    "unique_identifier_ratio","avg_identifier_length","snake_case_ratio",
    "keyword_density","hapax_legomena_count",
    "ast_node_count","ast_max_depth","ast_avg_branching_factor",
    "control_flow_ratio","exception_handling_ratio","return_statement_density",
    "syntactic_ngram_frequency","cyclomatic_complexity",
    "halstead_volume","halstead_difficulty","halstead_effort",
    "maintainability_index","cognitive_complexity","token_entropy",
    "ast_node_entropy","perplexity_score","burstiness","halstead_n1","halstead_n2","halstead_N1",              
    "halstead_N2","halstead_vocab","halstead_length",  
]


In [ ]:
# Robust tuning + evaluation + feature-importance pipeline (Kaggle-ready)
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report
)
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Optional libraries
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

# === USER-SPECIFIC VARIABLES (change if needed) ===
TARGET_COL = "label"              # change if your label column name differs
FINAL_FEATURES = FINAL_FEATURES   # expects a Python list; already defined earlier
RESULT_DIR = "tuning_results"
os.makedirs(RESULT_DIR, exist_ok=True)
N_FOLDS = 5
N_ITER = 7                     # RandomizedSearchCV iterations per model15
N_PERM = 5                       # permutation importance repeats (test set)

# === Prepare X, y (train) and X_test, y_test (test) ===

X = df[FINAL_FEATURES].copy()
y = df[TARGET_COL].copy()
X_test = testdf[FINAL_FEATURES].copy()
y_test = testdf[TARGET_COL].copy()
#=======================================================
X = X.replace([np.inf, -np.inf], np.nan) 
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.mean())
X_test = X_test.fillna(X_test.mean())

assert X.shape[1] == len(FINAL_FEATURES), "Feature count mismatch"

print(f"Shape of train and test : {X.shape} ,  {X_test.shape}")
# === Models and hyperparameter search spaces ===
models_and_spaces = dict()

models_and_spaces["Logistic_L1"] = (
    Pipeline([("scaler", StandardScaler()), ("clf",
        LogisticRegression(solver="liblinear", max_iter=2000, random_state=RANDOM_STATE))]),
    {
        "clf__penalty": ["l1", "l2"],
        "clf__C": np.logspace(-3, 2, 20),
    }
)

models_and_spaces["RandomForest"] = (
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {
        "n_estimators": [200, 400, 800],
        "max_depth": [None, 6, 10, 15],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5]
    }
)

if XGBClassifier is not None:
    models_and_spaces["XGBoost"] = (
        XGBClassifier( eval_metric="logloss",tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1),
        {
            "n_estimators": [200, 400, 800],
            "max_depth": [3, 6, 10],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "subsample": [0.6, 0.8, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0]
        }
    )

if CatBoostClassifier is not None:
    models_and_spaces["CatBoost"] = (
        CatBoostClassifier(verbose=0 ,random_state=RANDOM_STATE),
        {
            "iterations": [200, 400, 800],
            "depth": [4, 6, 10],
            "learning_rate": [0.01, 0.05, 0.1],
            "l2_leaf_reg": [1, 3, 10]
        }
    )

models_and_spaces["MLP"] = (
    Pipeline([("scaler", StandardScaler()), ("clf", MLPClassifier(max_iter=1500, random_state=RANDOM_STATE))]),
    {
        "clf__hidden_layer_sizes": [(128,64), (256,128), (64,32)],
        "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__learning_rate_init": [1e-4, 1e-3, 1e-2]
    }
)

# === Cross-validation setup used by RandomizedSearchCV ===
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# === Storage for results ===
tuned_models = dict()
eval_summary = []
feature_importances = pd.DataFrame(index=FINAL_FEATURES)

# === Function to compute model importances on test set ===
def compute_importances_on_test(model, X_test, y_test, feature_names):
    # If pipeline, unwrap core estimator for attribute checks
    core = model
    if hasattr(model, "named_steps"):
        # assume last step is estimator
        core = list(model.named_steps.values())[-1]

    # 1) Use feature_importances_ if present
    if hasattr(core, "feature_importances_"):
        imp = core.feature_importances_
        imp = np.array(imp, dtype=float)
        if imp.sum() > 0: imp = imp / imp.sum()
        return imp

    # 2) Use coef_ for linear models
    if hasattr(core, "coef_"):
        coef = np.abs(core.coef_)
        if coef.ndim > 1:
            coef = coef.mean(axis=0)
        imp = np.array(coef, dtype=float)
        if imp.sum() > 0: imp = imp / imp.sum()
        return imp

    # 3) Fallback: permutation importance on test set
    try:
        perm = permutation_importance(model, X_test, y_test, n_repeats=N_PERM, random_state=RANDOM_STATE, n_jobs=-1, scoring="f1")
        imp = perm.importances_mean
        # shift to non-negative
        if imp.min() < 0:
            imp = imp - imp.min()
        if imp.sum() > 0:
            imp = imp / imp.sum()
        return imp
    except Exception as e:
        # ultimate fallback: zero vector
        print("Permutation importance failed:", e)
        return np.zeros(len(feature_names), dtype=float)
# Before the loop
X_tune, _, y_tune, _ = train_test_split(X, y, train_size=100000, stratify=y, random_state=RANDOM_STATE)

# === Run RandomizedSearchCV per model, evaluate on test set, collect importances ===
for name, (estimator, param_dist) in models_and_spaces.items():
    print(f"\n--- Tuning model: {name} ---")
        
    # RandomizedSearchCV
    rs = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring="f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )
    if name == "Logistic_L1":
        LOGISTIC_N_ITER = 5   # instead of 15
        LOGISTIC_N_FOLDS = 3  # instead of 5
        cv_logistic = StratifiedKFold(n_splits=LOGISTIC_N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        rs_logistic = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_dist,
            n_iter=LOGISTIC_N_ITER,
            scoring="f1",
            cv=cv_logistic,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        rs = rs_logistic
    # In the loop, use X_tune/y_tune for rs.fit()
    rs.fit(X_tune, y_tune)
    # After search, refit best on full X/y
    best_full = rs.best_estimator_
    best_full.fit(X, y)  # quick for trees, acceptable for logistic
    tuned_models[name] = best_full
   
    # Evaluate on test set
    y_pred = best_full.predict(X_test)
    metrics = {
        "model": name,
        "best_params": rs.best_params_,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0)
    }
    # add ROC-AUC if binary and probabilities available
    try:
        if len(np.unique(y_test)) == 2 and hasattr(best, "predict_proba"):
            probs = best.predict_proba(X_test)[:,1]
            metrics["roc_auc"] = roc_auc_score(y_test, probs)
    except Exception:
        pass

    eval_summary.append(metrics)

    # Feature importances on test set
    imp = compute_importances_on_test(best_full, X_test, y_test, FINAL_FEATURES)
    feature_importances[name] = imp

    # Save classification report
    cr = classification_report(y_test, y_pred, output_dict=True)
    pd.DataFrame(cr).to_csv(os.path.join(RESULT_DIR, f"classif_report_{name}.csv"))

    print(f"Model {name} - test f1: {metrics['f1']:.4f}, accuracy: {metrics['accuracy']:.4f}")

# === Aggregate importances and save ===
feature_importances["mean_importance"] = feature_importances.mean(axis=1)
feature_importances = feature_importances.sort_values("mean_importance", ascending=False)
feature_importances.to_csv(os.path.join(RESULT_DIR, "feature_importances_tuned_models.csv"))

# === Save evaluation summary ===
eval_df = pd.DataFrame(eval_summary).sort_values("f1", ascending=False)
eval_df.to_csv(os.path.join(RESULT_DIR, "model_evaluation_summary.csv"), index=False)

# === Plotting ===
TOP_K = min(20, len(FINAL_FEATURES))
plt.rcParams.update({'figure.max_open_warning': 0})

# Per-model top-K importance bars
for col in [c for c in feature_importances.columns if c not in ("mean_importance")]:
    if col == "mean_importance": continue
    top = feature_importances[col].sort_values(ascending=False).head(TOP_K)
    plt.figure(figsize=(10,6))
    plt.title(f"{name} - Top {TOP_K} importances ({col})")
    plt.barh(top.index[::-1], top.values[::-1])
    plt.xlabel("Normalized importance")
    plt.tight_layout()
    plt.show()
    # plt.savefig(os.path.join(RESULT_DIR, f"importance_{col}.png"))
    plt.close()

# Aggregated importance bar chart (mean across models)
agg = feature_importances["mean_importance"].sort_values(ascending=False).head(TOP_K)
plt.figure(figsize=(12,7))
plt.title(f"Aggregated mean importance (top {TOP_K}) — tuned models")
plt.barh(agg.index[::-1], agg.values[::-1])
plt.xlabel("Mean normalized importance across tuned models")
plt.tight_layout()
plt.show()
# plt.savefig(os.path.join(RESULT_DIR, f"importance_aggregated_tuned_top{TOP_K}.png"))
plt.close()

# Histogram of aggregated importances
plt.figure(figsize=(7,4))
plt.title("Distribution of aggregated mean importances")
plt.hist(feature_importances["mean_importance"].values, bins=30)
plt.tight_layout()
# plt.savefig(os.path.join(RESULT_DIR, "hist_aggregated_importances.png"))
plt.show()
plt.close()

# === Final selected features (by aggregated importance) ===
FINAL_TOP_K = 15
final_selected = feature_importances.head(FINAL_TOP_K).index.tolist()
pd.Series(final_selected, name="selected_feature").to_csv(os.path.join(RESULT_DIR, f"final_selected_top{FINAL_TOP_K}.csv"), index=False)

print(f"\nTuning & evaluation finished. Results saved to '{RESULT_DIR}/'.")
print("Top models by F1 on test set:")
print(eval_df[["model","f1","accuracy","precision","recall"]].head(10))
print(f"\nFinal top-{FINAL_TOP_K} selected features saved as CSV.")


In [ ]:
import os
import pandas as pd

def print_all_classification_reports(
    result_dir,
    round_digits=4,
    sort_models=True
):
    """
    Traverse directory, load all classification report CSVs,
    and print them in a clean, readable format.
    """
    files = [
        f for f in os.listdir(result_dir)
        if f.startswith("classif_report_") and f.endswith(".csv")
    ]

    if not files:
        print("No classification report CSVs found.")
        return

    if sort_models:
        files = sorted(files)

    for fname in files:
        model_name = fname.replace("classif_report_", "").replace(".csv", "")
        path = os.path.join(result_dir, fname)

        print("\n" + "=" * 70)
        print(f"📊 Classification Report — {model_name}")
        print("=" * 70)

        df = pd.read_csv(path, index_col=0)

        # Convert all numeric values safely
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="ignore")

        # Round numeric values
        df = df.round(round_digits)

        # Pretty print
        print(df.to_string())

        # Highlight key metrics
        if "weighted avg" in df.columns:
            try:
                f1_weighted = df.loc["f1-score", "weighted avg"]
                acc = df.loc["accuracy", "weighted avg"]
                print("\nSummary:")
                print(f"  • Accuracy (weighted): {acc}")
                print(f"  • F1-score (weighted): {f1_weighted}")
            except Exception:
                pass

    print("\n✅ Finished printing all classification reports.")
RESULT_DIR = "/kaggle/working/tuning_results"

print_all_classification_reports(RESULT_DIR)


In [ ]:
path =  "/kaggle/working/tuning_results/final_selected_top15.csv"
rdf = pd.read_csv(path, index_col=0)
rdf.head(15)

In [ ]:
path =  "/kaggle/working/tuning_results/model_evaluation_summary.csv"
rdf = pd.read_csv(path, index_col=0)
rdf.head()
bestp = rdf['best_params']
for b in bestp :
    import pprint
    pprint.pprint(b)

In [ ]:
path = "/kaggle/working/tuning_results/feature_importances_tuned_models.csv"
# Sort by 'mean_importancd' in descending order and take top 20 
top20 = rdf.sort_values(by="mean_importance", ascending=False)
top20.head(20)

In [11]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Example DataFrame
# X -> feature matrix (35 features)
# y -> target variable

# Separate features and target
X = df.drop(columns=['label','language','code'])   # replace 'target' with your target column name
y = df['label']

# Scale features to be non-negative (required for chi-square)
scaler = MinMaxScaler()
X = X.replace([np.inf, -np.inf], np.nan) 
X = X.fillna(X.mean())
X_scaled = scaler.fit_transform(X)

# Apply Chi-Square feature selection
chi_selector = SelectKBest(score_func=chi2, k=15)
X_selected = chi_selector.fit_transform(X_scaled, y)

# Get selected feature names
selected_features = X.columns[chi_selector.get_support()]

# Create new DataFrame with selected features
X_reduced = pd.DataFrame(X_selected, columns=selected_features)

print("Selected Features:")
print(selected_features)


Selected Features:
Index(['loc_comments', 'comment_ratio', 'unique_identifier_ratio',
       'avg_identifier_length', 'snake_case_ratio', 'hapax_legomena_count',
       'ast_node_count', 'ast_max_depth', 'ast_avg_branching_factor',
       'control_flow_ratio', 'exception_handling_ratio',
       'syntactic_ngram_frequency', 'cyclomatic_complexity',
       'maintainability_index', 'ast_node_entropy'],
      dtype='object')


In [12]:
chi_scores = pd.DataFrame({
    'Feature': X.columns,
    'Chi2 Score': chi_selector.scores_
}).sort_values(by='Chi2 Score', ascending=False)

print(chi_scores)


                      Feature    Chi2 Score
3               comment_ratio  13892.946353
13              ast_max_depth   7165.901074
15         control_flow_ratio   6093.186892
9            snake_case_ratio   5078.819311
18  syntactic_ngram_frequency   4997.430677
7     unique_identifier_ratio   3244.290093
19      cyclomatic_complexity   1102.089466
11       hapax_legomena_count   1080.636890
8       avg_identifier_length    737.576838
14   ast_avg_branching_factor    729.487244
12             ast_node_count    603.228928
1                loc_comments    159.666610
26           ast_node_entropy    148.390501
16   exception_handling_ratio    119.493955
23      maintainability_index     81.037377
0                   loc_total     59.113861
33             halstead_vocab     52.986905
32                halstead_N2     48.858127
20            halstead_volume     27.398163
10            keyword_density     21.922660
21        halstead_difficulty     17.707571
25              token_entropy   

In [13]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_classif


# Apply Mutual Information
mi_selector = SelectKBest(score_func=mutual_info_classif, k=15)
X_selected = mi_selector.fit_transform(X, y)

# Get selected feature names
selected_features = X.columns[mi_selector.get_support()]

# Create reduced dataframe
X_reduced = pd.DataFrame(X_selected, columns=selected_features)

print("Selected Features:")
print(selected_features)


Selected Features:
Index(['loc_comments', 'loc_blank', 'comment_ratio', 'avg_line_length',
       'max_line_length', 'unique_identifier_ratio', 'avg_identifier_length',
       'hapax_legomena_count', 'ast_node_count', 'ast_max_depth',
       'ast_avg_branching_factor', 'control_flow_ratio',
       'cyclomatic_complexity', 'cognitive_complexity', 'ast_node_entropy'],
      dtype='object')


In [15]:
mi_scores = pd.DataFrame({
    'Feature': X.columns,
    'MI Score': mi_selector.scores_
}).sort_values(by='MI Score', ascending=False)

mi_scores.head(35)

,Feature,MI Score
8,avg_identifier_length,0.295137
13,ast_max_depth,0.251710
26,ast_node_entropy,0.249504
12,ast_node_count,0.249106
14,ast_avg_branching_factor,0.248184
19,cyclomatic_complexity,0.241663
7,unique_identifier_ratio,0.240905
11,hapax_legomena_count,0.213233
2,loc_blank,0.201731
24,cognitive_complexity,0.190498


In [17]:
# Combine X and y for correlation
df_corr = pd.concat([X, y], axis=1)

# Compute correlation matrix
correlation_matrix = df_corr.corr()

# Get correlation of features with target
target_corr = correlation_matrix['label'].drop('label')

# Select top 15 features based on absolute correlation
top_15_features = target_corr.abs().sort_values(ascending=False).head(15).index

# Reduced dataframe
X_reduced = X[top_15_features]

print("Selected Features:")
print(top_15_features)


Selected Features:
Index(['ast_max_depth', 'ast_avg_branching_factor', 'control_flow_ratio',
       'unique_identifier_ratio', 'comment_ratio', 'cyclomatic_complexity',
       'syntactic_ngram_frequency', 'hapax_legomena_count',
       'avg_identifier_length', 'ast_node_count', 'maintainability_index',
       'halstead_vocab', 'loc_comments', 'halstead_N2', 'snake_case_ratio'],
      dtype='object')


In [24]:
corr_scores = pd.DataFrame({
    'Feature': target_corr.index,
    'Correlation': target_corr.values
}).sort_values(by='Correlation', key=abs, ascending=False)

corr_scores.head(35)

,Feature,Correlation
13,ast_max_depth,-0.573195
14,ast_avg_branching_factor,-0.532796
15,control_flow_ratio,-0.436848
7,unique_identifier_ratio,-0.435513
3,comment_ratio,0.352442
19,cyclomatic_complexity,-0.311897
18,syntactic_ngram_frequency,-0.286204
11,hapax_legomena_count,-0.261630
8,avg_identifier_length,-0.243768
12,ast_node_count,-0.239855


In [23]:
corr_matrix = X.corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.7)]
X_filtered = X.drop(columns=to_drop)
print(to_drop)

['loc_blank', 'ast_avg_branching_factor', 'cyclomatic_complexity', 'halstead_volume', 'cognitive_complexity', 'ast_node_entropy', 'perplexity_score', 'halstead_n1', 'halstead_n2', 'halstead_N1', 'halstead_N2', 'halstead_vocab']


# Combining all 

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, mutual_info_classif

features = X.columns
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

chi_scores, _ = chi2(X_scaled, y)
chi_rank = pd.Series(chi_scores, index=features).rank(ascending=False)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

chi_scores, _ = chi2(X_scaled, y)
chi_rank = pd.Series(chi_scores, index=features).rank(ascending=False)
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_rank = pd.Series(mi_scores, index=features).rank(ascending=False)
corr_scores = X.corrwith(y).abs()
corr_rank = corr_scores.rank(ascending=False)


In [26]:
combined_rank = chi_rank + mi_rank + corr_rank
top_15_features = combined_rank.sort_values().head(15).index
X_reduced = X[top_15_features]

print("Top 15 Selected Features:")
print(top_15_features)
ranking_df = pd.DataFrame({
    'Chi-Square Rank': chi_rank,
    'MI Rank': mi_rank,
    'Correlation Rank': corr_rank,
    'Combined Rank': combined_rank
}).sort_values(by='Combined Rank')

ranking_df.head(35)

Top 15 Selected Features:
Index(['ast_max_depth', 'unique_identifier_ratio', 'ast_avg_branching_factor',
       'control_flow_ratio', 'comment_ratio', 'avg_identifier_length',
       'cyclomatic_complexity', 'ast_node_count', 'hapax_legomena_count',
       'ast_node_entropy', 'loc_comments', 'syntactic_ngram_frequency',
       'maintainability_index', 'snake_case_ratio', 'halstead_volume'],
      dtype='object')


,Chi-Square Rank,MI Rank,Correlation Rank,Combined Rank
ast_max_depth,2.0,2.0,1.0,5.0
unique_identifier_ratio,6.0,7.0,4.0,17.0
ast_avg_branching_factor,10.0,5.0,2.0,17.0
control_flow_ratio,3.0,11.0,3.0,17.0
comment_ratio,1.0,12.0,5.0,18.0
avg_identifier_length,9.0,1.0,9.0,19.0
cyclomatic_complexity,7.0,6.0,6.0,19.0
ast_node_count,11.0,3.0,10.0,24.0
hapax_legomena_count,8.0,8.0,8.0,24.0
ast_node_entropy,13.0,4.0,16.0,33.0


In [28]:
ranking_df.head(20)

,Chi-Square Rank,MI Rank,Correlation Rank,Combined Rank
ast_max_depth,2.0,2.0,1.0,5.0
unique_identifier_ratio,6.0,7.0,4.0,17.0
ast_avg_branching_factor,10.0,5.0,2.0,17.0
control_flow_ratio,3.0,11.0,3.0,17.0
comment_ratio,1.0,12.0,5.0,18.0
avg_identifier_length,9.0,1.0,9.0,19.0
cyclomatic_complexity,7.0,6.0,6.0,19.0
ast_node_count,11.0,3.0,10.0,24.0
hapax_legomena_count,8.0,8.0,8.0,24.0
ast_node_entropy,13.0,4.0,16.0,33.0


In [ ]:
import pandas as pd

df = pd.read_csv("feature_importances_tuned_models.csv")

print(df.head())


#df.set_index('feature', inplace=True)   # use correct column name
df.columns = ['feature'] + list(df.columns[1:])
df.set_index('feature', inplace=True)  
df.drop(columns=['mean_importance'], inplace=True, errors='ignore')
ranked_df = df.rank(
    ascending=False,
    method='average'
)
ranked_df['combined_rank'] = ranked_df.mean(axis=1)
final_ranking = ranked_df.sort_values('combined_rank')

print(final_ranking.head(15))
final_ranking.reset_index().to_csv("final_feature_ranking.csv", index=False)


"""
feature,Logistic_L1,RandomForest,XGBoost,CatBoost,MLP,combined_rank
ast_node_count,7.0,4.0,5.0,2.0,1.0,3.8
comment_ratio,8.0,8.0,3.0,5.0,7.0,6.2
loc_blank,28.0,1.0,4.0,3.0,3.0,7.8
ast_avg_branching_factor,5.0,2.0,23.0,10.0,5.0,9.0
ast_max_depth,27.0,6.0,1.0,1.0,13.0,9.6
snake_case_ratio,17.0,10.0,2.0,8.0,28.0,13.0
halstead_N1,16.0,20.0,10.0,11.0,8.0,13.0
loc_comments,14.0,9.0,6.0,27.0,12.0,13.6
avg_line_length,13.0,7.0,9.0,4.0,35.0,13.6
avg_identifier_length,19.0,5.0,20.0,17.0,10.0,14.2
loc_total,18.0,19.0,13.0,15.0,6.0,14.2
return_statement_density,24.0,16.0,8.0,9.0,16.0,14.6
indentation_std_dev,30.0,17.0,16.0,7.0,9.0,15.8
halstead_n2,1.0,32.0,11.0,21.0,14.0,15.8
cognitive_complexity,9.0,15.0,18.0,18.0,22.0,16.4
halstead_n1,10.0,27.0,7.0,13.0,26.0,16.6
ast_node_entropy,26.0,3.0,34.0,12.0,15.0,18.0
halstead_N2,6.0,31.0,25.0,24.0,4.0,18.0
halstead_difficulty,2.0,21.0,19.0,30.0,21.0,18.6
control_flow_ratio,29.0,13.0,15.0,14.0,23.0,18.8
halstead_volume,15.0,25.0,12.0,31.0,11.0,18.8
hapax_legomena_count,20.0,14.0,14.0,16.0,32.0,19.2
maintainability_index,12.0,23.0,30.0,34.0,2.0,20.2
max_line_length,21.0,18.0,28.0,22.0,18.0,21.4
perplexity_score,3.0,28.0,27.0,25.0,27.0,22.0
halstead_effort,11.0,24.0,24.0,32.0,19.0,22.0
unique_identifier_ratio,32.0,12.0,33.0,6.0,29.0,22.4
cyclomatic_complexity,31.0,11.0,21.0,19.0,34.0,23.2
halstead_vocab,4.0,33.0,22.0,26.0,33.0,23.6
burstiness,23.0,22.0,32.0,23.0,20.0,24.0
halstead_length,25.0,30.0,17.0,28.0,30.0,26.0
token_entropy,22.0,26.0,35.0,33.0,24.0,28.0
syntactic_ngram_frequency,33.0,34.0,29.0,29.0,17.0,28.4
keyword_density,35.0,29.0,31.0,20.0,31.0,29.2
exception_handling_ratio,34.0,35.0,26.0,35.0,25.0,31.0

"""